# Projeto #4 - Planejamento de Capacidade na Nuvem

## Equipe

- Carlos Duarte - matr. 2527530
- Jonas de A. Luz Jr. - matr. 2519171

----

In [3]:
import os, re

import pandas as pd

## Objetivo
>
> Fonte: [Especificação do projeto 4](https://docs.google.com/document/d/13QL64Om-XBFfEqDDyZp8czq-vdWQG-e4mvE3uwRpi-c/edit?tab=t.0)

**Da expecificação original:**

- Arquitetar a infraestrutura de um serviço de blog (WordPress) na AWS.
- Desafio: Maximizar o RPS (Requests Per Second) suportado pelo serviço, sujeito às seguintes restrições:
  - Orçamento: O custo da sua camada de aplicação não pode exceder US$0.50/hora (preço On-Demand us-east-1).
  - Qualidade (SLO): Taxa de Erro < 1% e Latência P95 < 10000ms.
  - Componentes Fixos: O Banco de Dados e o Load Balancer são fornecidos pela "Arena" e não podem ser modificados.
- O trabalho é ajustar a camada de aplicação, escolhendo a melhor combinação de escalabilidade vertical (tamanho da máquina) e horizontal (quantidade de máquinas) para implantar o WordPress.

São fornecidos os *scripts base* para a implementação do WordPress, que podem ser encontrados na pasta `scripts` do projeto.

## Estratégia de Implementação

Nossa estratégia de implementação do trabalho foi a seguinte:

1. Converter os scripts originais para o formato de *batch* do PowerShell, uma vez que o trabalho foi realizado em ambiente de desenvolvimento Windows.
2. Levantar os custos operacionais das instâncias de teste, uma vez que o orçamento foi limitado a US$0.50/hora. Os custos oficiais da AWS foram consultados [no site oficial do serviço EC2](https://us-east-1.console.aws.amazon.com/ec2/home?region=us-east-1).
3. Definição de experimentos para teste de escalabilidade vertical e horizontal, com o objetivo de encontrar a melhor combinação de tamanho da máquina e quantidade de máquinas para implantar o WordPress.
4. Implementação de scripts de apoio com o objetivo de automatizar a execução dos experimentos e coleta de métricas.
5. Implementação de scripts de apoio para coleta de métricas e análise de resultados.
6. Realização de modificações na configuração da aplicação WordPress para otimização do desempenho.
7. Avaliação dos resultados dos experimentos e coleta de métricas.
8. Elaboração de relatório com os resultados dos experimentos e coleta de métricas e escolha da melhor combinação de tamanho da máquina e quantidade de máquinas para implantar o WordPress.

O detalhamento das etapas é descrito nas seções seguintes.

## Implementação

### Conversão dos Scripts para o PowerShell

Os scripts originais foram convertidos para o formato de *batch* do PowerShell, uma vez que o trabalho foi realizado em ambiente de desenvolvimento Windows.

Os novos scripts constam na pasta `scripts` do projeto, enquanto os scripts originais foram preservados na subpasta `scripts/_original_bash_scripts`.

Foram mantidos em formato bash os scripts que, na verdade, são transferidos para as instâncias de teste, guardados na subpasta `scripts/data_scripts`.

### Levantamento de Custos Operacionais

Os custos operacionais das instâncias de teste foram obtidos [no site oficial do serviço EC2](https://us-east-1.console.aws.amazon.com/ec2/home?region=us-east-1). Os dados extraídos da página de preços do EC2 foram consolidados em um DataFrame Pandas.

In [4]:
# Dados de tipos de instância baixados do site de preços oficial do EC2, com custo menos que US$0.50/hora
#
PRICES_DATA_PATH = "data/ec2-prices"
data_files = os.listdir(PRICES_DATA_PATH)
print(f"Encontrados {len(data_files)} arquivos. Iniciando extração de dados...")

df_prices = pd.DataFrame()
for csv_file in data_files:
    print(csv_file, end='... ')
    
    csv_partial_file = os.path.join(PRICES_DATA_PATH, csv_file)
    df_prices = pd.concat([df_prices, pd.read_csv(csv_partial_file)], ignore_index=True)

print(f"\nRegistradas {df_prices.shape[0]} linhas.")

Encontrados 7 arquivos. Iniciando extração de dados...
instancetypes-p1.csv... instancetypes-p2.csv... instancetypes-p3.csv... instancetypes-p4.csv... instancetypes-p5.csv... instancetypes-p6.csv... instancetypes-p7.csv... 
Registradas 339 linhas.


In [5]:
# Convertendo valor do custo e selecionando os tipos de instância candidatos.
#
PRICE_COLUMN = 'On-Demand Linux pricing'

float_extractor = lambda v: float(re.findall(r"[-+]?\d*\.\d+|\d+", str(v))[0])

df_prices['Cost'] = df_prices[PRICE_COLUMN].apply(float_extractor)
df_prices = df_prices[(df_prices['Cost'] > 0) & (df_prices['Cost'] <= 0.5)]

print(f"Filtradas {df_prices.shape[0]} linhas com custo menor que US$0.50/hora.")

Filtradas 327 linhas com custo menor que US$0.50/hora.


Com a lista de tipos de instância candidatos inicial, foram excluídas as famílias de tipos que não interessam ou não se aplicam ao problema, com base nas [descrições oficiais de cada tipo de instância](https://docs.aws.amazon.com/ec2/latest/instancetypes/instance-type-names.html), como as famílias iniciadas por `g`, voltadas para aceleração por GPU ou `h` e `d`, que utilizam armazenamento HDD, dentre outras.

Por esta regra, foram selecionados os tipos de instância `t3`, por ter sido utilizado como exemplo do problema, instâncias das famílias `c`, otimizadas para computação e algumas outras da família `r`, otimizadas para memória. Foram também removidas as famílias variantes, como, por exemplo, aquelas que especificam o processador -- `c5a`, que indica uso de AMD, ou `c6i` que especifica o uso de Intel, etc -- ou variantes que modificam o armazenamento ou rede. Assim, restaram para análise os tipos de instância `t3`, `c1`, `c3`, `c4`, `c5`, `m1` a `m5` e `r3`, `r4` e `r5`.

In [6]:
INSTANCE_TYPE_COLUMN = 'Instance type'

families = sorted(
    df_prices[INSTANCE_TYPE_COLUMN].str.split('.').str[0].unique()
)

print(f"Famílias existentes: {families}")

Famílias existentes: ['a1', 'c1', 'c3', 'c4', 'c5', 'c5a', 'c5ad', 'c5d', 'c5n', 'c6a', 'c6g', 'c6gd', 'c6gn', 'c6i', 'c6id', 'c6in', 'c7a', 'c7g', 'c7gd', 'c7gn', 'c7i', 'c7i-flex', 'c8a', 'c8g', 'c8gb', 'c8gd', 'c8gn', 'c8i', 'c8i-flex', 'd3', 'g4ad', 'g5g', 'g6f', 'h1', 'i3', 'i3en', 'i4g', 'i4i', 'i7i', 'i7ie', 'i8g', 'i8ge', 'im4gn', 'inf1', 'is4gen', 'm1', 'm2', 'm3', 'm4', 'm5', 'm5a', 'm5ad', 'm5d', 'm5dn', 'm5n', 'm5zn', 'm6a', 'm6g', 'm6gd', 'm6i', 'm6id', 'm6idn', 'm6in', 'm7a', 'm7g', 'm7gd', 'm7i', 'm7i-flex', 'm8a', 'm8g', 'm8gb', 'm8gd', 'm8gn', 'm8i', 'm8i-flex', 'r3', 'r4', 'r5', 'r5a', 'r5ad', 'r5b', 'r5d', 'r5dn', 'r5n', 'r6a', 'r6g', 'r6gd', 'r6i', 'r6id', 'r6idn', 'r6in', 'r7a', 'r7g', 'r7gd', 'r7i', 'r7iz', 'r8a', 'r8g', 'r8gb', 'r8gd', 'r8gn', 'r8i', 'r8i-flex', 't1', 't2', 't3', 't3a', 't4g', 'x2gd', 'x8g', 'z1d']


In [7]:
filter = lambda x: x.startswith('t3.') or x.split('.')[0] in ['c1', 'c3', 'm1', 'm2', 'm3', 'm4', 'm5', 'r3', 'r4', 'r5']

df_prices = df_prices[df_prices[INSTANCE_TYPE_COLUMN].apply(filter)]

print(f"Filtradas {df_prices.shape[0]} linhas com famílias de instância de interesse.")

Filtradas 31 linhas com famílias de instância de interesse.


Para cada uma das famílias candidatas, calculamos a quantidade máxima de instâncias que podem ser implantadas no orçamento de US$0.50/hora.

In [8]:
max_calculator = lambda x: int(0.5 / x)

df_prices.loc[:, 'Max Instances'] = df_prices['Cost'].apply(max_calculator)

In [9]:
df_prices [['Instance type', 'Cost', 'Max Instances']].sort_values(by='Max Instances', ascending=False)

,Instance type,Cost,Max Instances
2,t3.micro,0.0104,48
7,t3.small,0.0208,24
18,t3.medium,0.0416,12
20,m1.small,0.0440,11
40,m3.medium,0.0670,7
57,t3.large,0.0832,6
66,m1.medium,0.0870,5
79,m5.large,0.0960,5
83,m4.large,0.1000,5
91,c3.large,0.1050,4


A partir da tabela de preços, foram identificadas os tipos de instância candidatos para o experimento, tendo sido selecionadas as instâncias t3.micro (tipo base, mais barato e mais leve, utilizado como exemplo na especificação do trabalho), c5.large, c5.xlarge e c5.2xlarge (tipo premium, mais caro e mais pesado). Estas escolhas visavam permitir os testes de escalabilidade horizontal e vertical, conforme definido na especificação do trabalho. 

### Experimentos

Os experimentos foram padronizados para testar cada configuração com um certo número de usuários a serem simulados com o Locust.

| Cenário | Quantidade de Usuários | 
| --- | --- |
| Uso Mínimo | 100 | 
| Uso Baixo | 250 |
| Uso Médio | 500 |
| Uso Alto | 1000 |
| Uso Muito Alto | 2000 |

#### Fase 1  Escalabilidade Horizontal


Para a escalabilidade horizontal, apenas podemos considerar os tipos de instância que permitem se utilizar mais de uma instância. Neste caso, em vez de testes exaustivos aumentando a quantidade de instâncias de uma em uma, optamos por fazer uso de três amostras para cada tipo de instância, conforme o desempenho obtido.

Inicialmente, testamos a escalabilidade horizontal, aumentando o número de instâncias de *t3.micro*.